# LO7. Agent 단기 메모리

**목표**

- InMemorySaver checkpointer와 thread_id로 여러 턴에 걸쳐 대화 맥락을 유지합니다.
- thread_id를 바꾸면 저장된 맥락이 끊기는 것을 확인합니다.
- `get_state`로 thread별 누적 상태를 직접 들여다봅니다.

InMemorySaver는 상태를 프로세스 메모리에만 보관하므로 런타임을 재시작하면 대화가 모두 사라집니다(데모용). 운영에서는 SqliteSaver나 PostgresSaver로 교체합니다.

## 1단계: 패키지 설치 (버전 핀 고정)

In [ ]:
# 버전 핀 고정 (v1.0 기준). 강의 직전 최신 안정 버전 재확인 권장
!pip install -U "langchain==1.3.4" "langgraph==1.2.4" langchain-openai langchain-google-genai

In [ ]:
import os
from google.colab import userdata

# Colab 좌측 열쇠 메뉴에 한 번 등록하면 이후 모든 LO에서 재사용됩니다
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 로컬, IDE 실행 대안 (위 두 줄 대신 사용)
#   export OPENAI_API_KEY="sk-..." 로 환경변수를 설정하면 아래 한 줄로 충분합니다
# import os
# os.environ.setdefault("OPENAI_API_KEY", os.environ.get("OPENAI_API_KEY", ""))
# 또는 .env 파일을 쓰려면: from dotenv import load_dotenv; load_dotenv()

## 2단계: 모델과 도구 준비

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

# 주 모델. 강의 직전 최신 모델, 가격 재확인. 대안: google-genai:gemini-2.5-flash
model = init_chat_model("openai:gpt-5.4-mini")

@tool
def add(a: int, b: int) -> int:
    """두 정수를 더한다."""
    return a + b

**체크포인트**: `init_chat_model`로 모델을 초기화하고, `@tool`로 간단한 덧셈 도구를 만들었습니다. 메모리 동작 자체에는 도구가 필수는 아니지만, 에이전트 형태를 유지하기 위해 함께 둡니다.

## 3단계: checkpointer를 붙여 Agent 컴파일

In [ ]:
from langchain.agents import create_agent           # v1 권장 한 줄 에이전트
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()                       # 프로세스 메모리에 상태 저장 (데모용)

# create_agent에 checkpointer를 넘기면 매 호출마다 상태가 누적되고 복원됩니다
agent = create_agent(
    "openai:gpt-5.4-mini",                           # 강의 직전 최신 모델, 가격 재확인
    tools=[add],
    system_prompt="너는 친절한 한국어 비서다.",
    checkpointer=checkpointer,
)

**체크포인트**: `checkpointer=checkpointer` 한 줄이 단기 메모리의 전부입니다. 이 부품이 매 단계 상태(누적 메시지)를 저장하고, 다음 호출에서 같은 thread_id면 복원합니다.

## 4단계: 같은 thread_id로 멀티턴 대화 유지

In [ ]:
# config의 thread_id가 어느 대화를 이어 갈지 결정합니다
config = {"configurable": {"thread_id": "user-123"}}

# 첫 번째 턴: 이름을 알려 줍니다
r1 = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름은 앤디야. 기억해 줘."}]},
    config,                                          # 같은 thread_id로 상태 저장
)
print(r1["messages"][-1].content)

# 두 번째 턴: 직전 대화를 기억하는지 확인 (같은 thread_id)
r2 = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]},
    config,                                          # 저장된 메시지가 복원되고 누적됩니다
)
print(r2["messages"][-1].content)                    # "앤디"라고 답하면 단기 메모리 동작

# 누적 메시지가 실제로 쌓였는지 확인 (첫 턴 user, ai + 둘째 턴 user, ai = 4개 이상)
print(len(r2["messages"]))                           # 예: 4 (도구 호출이 있으면 더 많음)

**체크포인트**: 두 번째 응답이 "앤디"를 기억하면 같은 thread_id 안에서 메시지가 누적된 것입니다. 누적 메시지 수가 4개 이상이면 첫 턴과 둘째 턴이 한 상태로 이어졌다는 뜻입니다.

## 5단계: thread_id를 바꾸면 맥락이 끊긴다

In [ ]:
# thread_id가 다르면 저장된 맥락이 없는 새 대화입니다
new_config = {"configurable": {"thread_id": "user-999"}}

r3 = agent.invoke(
    {"messages": [{"role": "user", "content": "내 이름이 뭐였지?"}]},
    new_config,                                      # user-123의 기억은 여기에 없음
)
print(r3["messages"][-1].content)                    # 이름을 모른다고 답하면 정상

**체크포인트**: 같은 thread_id면 메시지가 누적되어 이름을 기억하고, thread_id를 바꾸면 저장된 맥락이 없어 기억하지 못합니다. 이것이 단기 메모리의 경계입니다.

> 주의: InMemorySaver는 런타임을 재시작하거나 노트북을 다시 실행하면 저장된 대화가 모두 사라집니다. 대화를 영구 보존하려면 SqliteSaver나 PostgresSaver로 교체합니다.

---

> **여기까지가 핵심입니다. 아래는 선택(심화)입니다.**

## 6단계: 저장된 상태 직접 들여다보기 (선택)

In [ ]:
# checkpointer에 저장된 현재 상태를 thread_id로 꺼내 봅니다
state = agent.get_state(config)                      # config는 user-123
for m in state.values["messages"]:
    # 메시지 종류(human/ai/tool)와 내용 앞부분만 출력
    print(type(m).__name__, repr(m.content[:40]))

# 변형 포인트: config를 new_config(user-999)로 바꾸면 메시지가 거의 비어 있어,
#   thread별로 상태가 따로 저장된다는 점을 눈으로 확인할 수 있습니다.

**체크포인트**: `get_state(config)`로 thread별 누적 상태를 직접 확인할 수 있습니다. thread_id를 바꿔 호출하면 서로 다른 메시지 목록이 돌아오며, 이것이 thread 격리가 동작하는 모습입니다.

### 실습 체크포인트

- [ ] 같은 thread_id로 두 번 호출했을 때 두 번째 응답이 첫 번째 대화를 기억한다
- [ ] thread_id를 바꾸면 같은 질문에 맥락 없이 답한다
- [ ] `get_state`로 꺼낸 메시지 목록이 thread_id마다 다르다(격리)는 것을 확인한다
- [ ] checkpointer를 붙이지 않으면(또는 config를 빼면) 매 호출이 독립적이어서 기억하지 못함을 설명할 수 있다